# GabayBigkas: bounded manual BuzzASR inference

This notebook processes one consented-adult MVP export bundle interactively. It is not a server, worker, tunnel, or remote-control endpoint. Never place Agora/AWS credentials, cookies, learner identities, or application secrets here.

In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 safetensors==0.6.2

In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA unavailable: select Runtime > Change runtime type > GPU'
GPU_NAME = torch.cuda.get_device_name(0)
print({'cuda_ready': True, 'gpu_name': GPU_NAME})

In [ ]:
import hashlib, io, json, re, time, wave, zipfile
from google.colab import files

SCHEMA_VERSION = 1
MODEL_ID = 'BuzzASR/filipino'
MODEL_REVISION = 'fb8cf0d93e2437f8d639549c67733ca9db10e055'
MAX_AUDIO_BYTES = 10 * 1024 * 1024
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one GabayBigkas ZIP bundle'
bundle_name, bundle_bytes = next(iter(uploaded.items()))
assert bundle_name.endswith('.zip'), 'The export must be a ZIP bundle'
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    assert set(archive.namelist()) == {'manifest.json', 'audio.wav'}, 'Unexpected bundle files'
    assert archive.getinfo('manifest.json').file_size <= 64 * 1024, 'Manifest is too large'
    assert archive.getinfo('audio.wav').file_size <= MAX_AUDIO_BYTES, 'Audio is too large'
    manifest = json.loads(archive.read('manifest.json'))
    audio = archive.read('audio.wav')
required = {'schema_version','job_id','audio_sha256','sample_rate','channels','model_id','model_revision','created_at','expires_at'}
assert set(manifest) == required, 'Manifest fields do not match schema'
assert manifest['schema_version'] == SCHEMA_VERSION, 'Schema mismatch'
assert re.fullmatch(r'[0-9a-fA-F-]{36}', manifest['job_id']), 'Invalid opaque job ID'
assert manifest['model_id'] == MODEL_ID, 'Wrong model ID'
assert manifest['model_revision'] == MODEL_REVISION, 'Wrong immutable model revision'
assert manifest['created_at'] <= int(time.time()) <= manifest['expires_at'], 'Bundle is expired or future-dated'
assert hashlib.sha256(audio).hexdigest() == manifest['audio_sha256'], 'Audio SHA-256 mismatch'
with wave.open(io.BytesIO(audio), 'rb') as wav:
    assert wav.getframerate() == manifest['sample_rate'] == 16000, 'Expected 16 kHz audio'
    assert wav.getnchannels() == manifest['channels'] == 1, 'Expected mono audio'
    assert wav.getsampwidth() == 2, 'Expected PCM16 audio'
open('/content/gabaybigkas-audio.wav', 'wb').write(audio)
print({'bundle_valid': True, 'job_id': manifest['job_id'], 'audio_bytes': len(audio), 'model_revision': MODEL_REVISION})

In [ ]:
from transformers import pipeline
asr = pipeline(
    'automatic-speech-recognition',
    model=MODEL_ID,
    revision=MODEL_REVISION,
    device=0,
    torch_dtype=torch.float16,
)
assert getattr(asr.model.config, '_commit_hash', MODEL_REVISION) == MODEL_REVISION, 'Loaded model revision mismatch'
print({'model_ready': True, 'model_id': MODEL_ID, 'model_revision': MODEL_REVISION})

In [ ]:
raw = asr('/content/gabaybigkas-audio.wav', return_timestamps='word')
words = []
for chunk in raw.get('chunks', []):
    start_s, end_s = chunk.get('timestamp', (None, None))
    assert start_s is not None and end_s is not None and 0 <= start_s <= end_s, 'Malformed model timestamp'
    words.append({'text': chunk.get('text', '').strip(), 'start_ms': round(start_s * 1000), 'end_ms': round(end_s * 1000)})
result = {
    'schema_version': SCHEMA_VERSION,
    'job_id': manifest['job_id'],
    'audio_sha256': manifest['audio_sha256'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'generated_at': int(time.time()),
    'text': raw.get('text', '').strip(),
    'segments': [],
    'words': words,
}
output_name = f"gabaybigkas-{manifest['job_id']}.result.json"
with open(output_name, 'w', encoding='utf-8') as output:
    json.dump(result, output, ensure_ascii=False, separators=(',', ':'))
print({'inference_complete': True, 'job_id': manifest['job_id'], 'word_count': len(words)})
files.download(output_name)

Import the downloaded result JSON through the authenticated GabayBigkas annotator interface. Close the Colab runtime afterward; do not expose it through a tunnel or leave it running as a service.